# Validating `HealthDataset-full-example.ttl`

This notebook runs the same SHACL check `tests/test_shacl_validation.py::test_dataset_conforms_to_merged_shacl` runs in CI, against the actual published example file sitting next to this notebook -- so you can see exactly how validation works and what it currently reports, without reading test-suite internals.

**The shapes graph** is `docs/schema/health_dcat_ap_plus.merged-shacl.ttl` -- ONE combined SHACL profile: HealthDCAT-AP's real, official upstream shapes for the Dataset/Distribution/Agent/Catalog side, plus this schema's own generated shapes for everything else (Activity/Association/Attribution/Entity). See `scripts/gen_merged_shacl.py`'s own docstring for why a naive union of the two doesn't work and what has to be filtered out.

**The tricky part**: `pyshacl` never dereferences external URIs. Our example references real HealthDCAT-AP vocabulary terms (e.g. `healthcategories/PHDR`, `standard/FHIR`) purely by IRI -- exactly what a real dataset publisher would do. Without the *describing* triples for those terms (`skos:Concept`, `skos:inScheme`, ...), `pyshacl` has no way to check `skos:inScheme`/`sh:class` constraints against them and would report false violations. So this notebook supplies those triples the same way the real test does: extracted on demand from HealthDCAT-AP's own real, cached vocabulary catalogue -- **not** added to the example file itself (a data publisher never asserts facts about *someone else's* vocabulary terms in their own dataset).

In [1]:
import sys
from pathlib import Path

import pyshacl
from rdflib import Graph

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
sys.path.insert(0, str(REPO_ROOT / "tests"))

import gen_merged_shacl
# Reuses the real project helpers rather than reimplementing them here, so
# this notebook can never silently drift from what the test suite actually
# checks -- see their own docstrings in tests/test_shacl_validation.py.
from test_shacl_validation import (
    _official_vocabulary_catalogue_graph,
    _real_vocabulary_terms_graph,
    _referenced_terms_closure_graph,
    _violation_signatures,
)

EXAMPLE_PATH = REPO_ROOT / "examples" / "HealthDataset-full-example.ttl"

## 1. Load the published example as-is

In [2]:
example_graph = Graph()
example_graph.parse(str(EXAMPLE_PATH), format="turtle")
print(f"{EXAMPLE_PATH.name}: {len(example_graph)} triples")

HealthDataset-full-example.ttl: 118 triples


## 2. Supply real vocabulary-membership triples for the terms this example actually references

`_official_vocabulary_catalogue_graph()` loads HealthDCAT-AP's own real, cached catalogue (the same one Sciensano's official hosted validator preloads for every submission). `_referenced_terms_closure_graph` then extracts *only* the describing triples for whatever the example actually references -- automatically, no hardcoded term list to maintain -- which is both correct and far faster than merging the whole ~200k-triple catalogue into the validation run.

In [3]:
catalogue = _official_vocabulary_catalogue_graph()
referenced_terms = _referenced_terms_closure_graph(example_graph, catalogue)
dpv_terms = _real_vocabulary_terms_graph()  # DPV isn't part of HealthDCAT-AP's own catalogue

data_graph = example_graph + referenced_terms + dpv_terms
print(f"Referenced-term closure: {len(referenced_terms)} triples")
print(f"DPV snippet: {len(dpv_terms)} triples")
print(f"Full data graph for validation: {len(data_graph)} triples")

Referenced-term closure: 9207 triples
DPV snippet: 7 triples
Full data graph for validation: 9330 triples


## 3. Build the merged shapes graph

Regenerated fresh in-memory here (needs the sibling `repos/healthdcat-ap` clone -- see README.md). This is the exact same shapes graph committed at `docs/schema/health_dcat_ap_plus.merged-shacl.ttl`; if you don't have that clone, load the committed file directly instead:
```python
shapes_graph = Graph()
shapes_graph.parse(str(REPO_ROOT / "docs" / "schema" / "health_dcat_ap_plus.merged-shacl.ttl"), format="turtle")
```

In [4]:
shapes_graph = gen_merged_shacl.build_merged_shapes_graph()
print(f"Merged shapes graph: {len(shapes_graph)} triples")

Merged shapes graph: 2686 triples


## 4. Validate

In [5]:
conforms, results_graph, results_text = pyshacl.validate(
    data_graph,
    shacl_graph=shapes_graph,
    data_graph_format="turtle",
    inference="none",
    advanced=True,
)
print(f"conforms = {conforms}")
signatures = _violation_signatures(results_graph)
print(f"{len(signatures)} distinct violation signature(s) (severity, constraint component, path):")
for sig in sorted(signatures):
    print(f"  {sig}")

conforms = False
2 distinct violation signature(s) (severity, constraint component, path):
  ('Violation', 'ClassConstraintComponent', 'type')
  ('Violation', 'MinCountConstraintComponent', 'value')


## 5. What's left, and why

As of this notebook, the example converges to exactly the same two known signatures `KNOWN_MERGED_SHAPES_VIOLATIONS` in `tests/test_shacl_validation.py` documents -- both individually traced to a specific node and shape, neither a bug in our own data:

- **`type`** (Violation, `rdf:type`) -- a real, pre-existing `dcat-ap-plus` schema quirk: `ClassifierMixin`'s `rdf_type` slot maps to the bare `rdf:type` predicate, the same one that asserts a node's own class -- so every `prov:Activity`/`prov:Entity` node spuriously "has a value" for `rdf_type` (its own class) that fails the `schema:DefinedTerm` check. Traced to a real LinkML `ShaclGenerator` bug, filed as [linkml/linkml#3931](https://github.com/linkml/linkml/issues/3931).
- **`value`** (Violation, `prov:value`) -- `QualitativeAttribute`'s own required `prov:value` slot bleeds onto every `prov:Entity`-typed node via a shared `class_uri` in `dcat-ap-plus`'s own base schema -- fires here on the population Entity node, which has no `prov:value` of its own. Same underlying LinkML `ShaclGenerator` mechanism as `type` above (shapes merging by `class_uri`), filed separately as [linkml/linkml#3932](https://github.com/linkml/linkml/issues/3932).

`dct:source`, `dct:conformsTo`, `dqv:hasQualityAnnotation`, and **`title`** used to be on this list too -- all four fixed for real, not worked around (a bare-URI port-script fix, a real vocabulary-term swap, a missing `inlined_as_list` fix, and -- for `title` -- a missing `xsd:` prefix redeclaration in this schema's own `prefixes:` block). `title` in particular had been misdiagnosed for months as an `rdflib`/`pyshacl` untyped-literal interop quirk; the real cause turned out to be this schema's own workaround for a separate, already-known LinkML prefix-propagation gap (imported schemas' prefixes don't reach `gen-shacl`/`gen-owl` output) simply never covering `xsd:` -- every `sh:datatype` constraint generated from a builtin XSD-backed range (`string`, `date`, `boolean`, ...) came out as the *unexpanded CURIE string* `"xsd:string"` used directly as a URIRef, which no real literal could ever satisfy. See `docs/architecture-verification.md` for the full misdiagnosis-then-correction narrative. Neither of the two remaining findings is fixable without touching `dcat-ap-plus`/LinkML themselves -- both filed upstream, waiting on maintainer response.

If you see anything else here, either something regressed, or the fixture/schema changed on purpose and `tests/test_shacl_validation.py`'s allowlists need updating to match -- see that file's own `_assert_known_violations` for the exact same check this notebook just ran by hand.

In [6]:
print(results_text)

Validation Report
Conforms: False
Results (21):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class <schema:DefinedTerm> ; sh:description Literal("The slot to specify the ontology class that is instantiated by an entity.") ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:IRI ; sh:order Literal("10", datatype=xsd:integer) ; sh:path rdf:type ]
	Focus Node: <https://example.org/population/regional-cancer-patients-2024>
	Value Node: prov:Entity
	Result Path: rdf:type
	Message: Value does not have class <schema:DefinedTerm>
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class <schema:DefinedTerm> ; sh:description Literal("The slot to specify the ontology class that is instantiated by an entity.") ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:IRI ; sh:orde